<a href="https://colab.research.google.com/github/yerikim15/AI-Mini-Project---Animal-Subspecies/blob/main/PAI_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Large
from google.colab import drive

# 1. Mount Google Drive so Colab can read your folders
drive.mount('/content/drive')

# 2. Double-check that your cloud T4 GPU is active
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
GPU Available:  []


In [ ]:
# Cloud paths pointing to your exact folder name 'animal_dataset gp18'
TRAIN_DIR = '/content/drive/MyDrive/animal_dataset gp18/dataset/train'
VAL_DIR = '/content/drive/MyDrive/animal_dataset gp18/dataset/val'
TEST_DIR = '/content/drive/MyDrive/animal_dataset gp18/dataset/test'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load the training, validation, and testing split folders
try:
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    val_dataset = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )
    test_dataset = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
    )

    # Get the number of classes automatically
    NUM_CLASSES = len(train_dataset.class_names)
    print(f"\nSuccessfully loaded {NUM_CLASSES} classes: {train_dataset.class_names}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please ensure the directories exist and are correctly named in your Google Drive.")
    print(f"Expected paths: ")
    print(f"  Train: {TRAIN_DIR}")
    print(f"  Validation: {VAL_DIR}")
    print(f"  Test: {TEST_DIR}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Found 404 files belonging to 18 classes.
Found 80 files belonging to 18 classes.
Found 105 files belonging to 18 classes.

Successfully loaded 18 classes: ['eagle_bald', 'eagle_golden', 'eagle_harpy', 'eagle_steppe', 'leopard_african', 'leopard_amur', 'leopard_clouded', 'leopard_snow', 'lion_african', 'lion_asiatic', 'lion_white', 'tiger_bengal', 'tiger_indochinese', 'tiger_siberian', 'tiger_sumatran', 'wolf_arctic', 'wolf_ethiopian', 'wolf_grey']


In [ ]:
def build_transfer_learning_model(model_name):
    # 1. Select the pre-trained architecture base
    if model_name == 'ResNet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    elif model_name == 'DenseNet121':
        base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    elif model_name == 'MobileNetV3':
        base_model = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # 2. Freeze pre-trained weights so we keep their learned features
    base_model.trainable = False

    # 3. Create your custom classification head for your 18 classes
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax') # Automatically handles 18 classes now!
    ])

    # 4. Compile with accuracy, precision, and recall metrics
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )
    return model

In [ ]:
history_records = {}
training_times = {}
model_list = ['ResNet50', 'DenseNet121', 'MobileNetV3']

for name in model_list:
    print(f"\n================ STARTING TRAINING FOR {name} ================\n")
    current_model = build_transfer_learning_model(name)

    start_time = time.time()

    # Train for the strict 50 epochs required by your rubric brief
    history = current_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=50
    )

    end_time = time.time()
    elapsed_time = end_time - start_time

    # Save statistics
    training_times[name] = elapsed_time
    history_records[name] = history.history

    # Save the files directly to your cloud directory so your analyst can use them
    current_model.save(f'{name}_subspecies_model.keras')
    print(f"\n Finished training {name} in {elapsed_time/60:.2f} minutes!")


================ STARTING TRAINING FOR ResNet50 ================

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 101s 7s/step - accuracy: 0.3465 - loss: 2.5168 - precision: 0.4785 - recall: 0.2203 - val_accuracy: 0.5375 - val_loss: 1.4603 - val_precision: 0.6944 - val_recall: 0.3125
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 85s 7s/step - accuracy: 0.6386 - loss: 1.2541 - precision: 0.7654 - recall: 0.4926 - val_accuracy: 0.6125 - val_loss: 1.2380 - val_precision: 0.8780 - val_recall: 0.4500
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 78s 6s/step - accuracy: 0.7698 - loss: 0.8202 - precision: 0.8728 - recall: 0.6114 - val_accuracy: 0.5875 - val_loss: 1.2906 - val_precision: 0.8542 - val_recall: 0.5125
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 89s 7s/step - accuracy: 0.8267 - loss: 0.6119 - precision: 0.9052 - recall: 0.7327 - val_accuracy: 0.6375 - val_loss: 1.2701 - val_precision: 0.7857 - val_recall: 0.5500
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 85s 7s/step

In [ ]:
import tensorflow as tf

print("\n================ FIXING FINE-TUNING AND SAVING ================\n")

# 1. Load the healthy model you already saved
backup_model = tf.keras.models.load_model('DenseNet121_subspecies_model.keras')

# 2. Unfreeze the layers
backup_model.layers[0].trainable = True

# 3. Recompile with a slow learning rate
backup_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

# 4. Quick 5-epoch training loop (cut down from 10 so it's lightning fast)
history = backup_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# 5. Save it with the CORRECT variable name
backup_model.save('Tuned_DenseNet121_subspecies_model.keras')
print("\n FIXED! The fine-tuned model file is now saved successfully!")


================ FIXING FINE-TUNING AND SAVING ================

Epoch 1/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 415s 25s/step - accuracy: 0.1658 - loss: 4.1200 - precision: 0.1583 - recall: 0.0941 - val_accuracy: 0.2000 - val_loss: 3.2371 - val_precision: 0.1714 - val_recall: 0.0750
Epoch 2/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 372s 25s/step - accuracy: 0.2475 - loss: 3.9186 - precision: 0.2618 - recall: 0.1510 - val_accuracy: 0.2125 - val_loss: 3.1477 - val_precision: 0.2000 - val_recall: 0.0875
Epoch 3/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 314s 24s/step - accuracy: 0.2426 - loss: 3.5775 - precision: 0.2925 - recall: 0.1832 - val_accuracy: 0.2500 - val_loss: 3.0229 - val_precision: 0.2778 - val_recall: 0.1250
Epoch 4/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 329s 25s/step - accuracy: 0.2921 - loss: 3.2150 - precision: 0.3833 - recall: 0.2277 - val_accuracy: 0.2875 - val_loss: 2.8817 - val_precision: 0.4103 - val_recall: 0.2000
Epoch 5/5
13/13 ━━━━━━━━━━━━━━━━━━━━ 307s 24s/step - accuracy: 0.3292 - loss: 2.9585 - precision: 

In [ ]:
import shutil

# The path to your shared team folder
shared_drive_folder = '/content/drive/MyDrive/animal_dataset gp18/'

# Copying the files from temporary storage into your shared Drive folder
shutil.copy('ResNet50_subspecies_model.keras', shared_drive_folder)
shutil.copy('DenseNet121_subspecies_model.keras', shared_drive_folder)
shutil.copy('MobileNetV3_subspecies_model.keras', shared_drive_folder)
shutil.copy('Tuned_DenseNet121_subspecies_model.keras', shared_drive_folder)

print("🎉 Success! The 4 models are now permanently saved in your shared Google Drive folder!")

🎉 Success! The 4 models are now permanently saved in your shared Google Drive folder!
